In [ ]:
import pandas as pd

df = pd.read_csv("/content/IMDB Dataset.csv",encoding ='latin1')
print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [ ]:
df['sentiment'] = df['sentiment'].map({'positive':1,'negative':0})

In [ ]:
import re

negation_words = [
    "Not Good","Not bad","not great","don't like","didn't like","never liked","wasn't good", "no good"
    ]

def clean_text(text):
  text = text.lower()

  text = re.sub(r"[^a-zAA-Z\s']"," ",text)

  for phrase in negation_words:
    text = text.replace(phrase,phrase.replace(" ",""))

  return text

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train , y_test = train_test_split(df['review'],df['sentiment'],test_size = 0.2,random_state = 42)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 20000
max_len = 250

tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_len,
    padding='post'
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_len,
    padding='post'
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(vocab_size,128,input_length=max_len),
    LSTM(128,dropout = 0.3,recurrent_dropout = 0.3),
    Dense(64,activation='relu'),
    Dropout(0.3),
    Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(X_train_pad,y_train,validation_split=0.2,epochs=5,batch_size=64)


Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 435s 862ms/step - accuracy: 0.5823 - loss: 0.6581 - val_accuracy: 0.6258 - val_loss: 0.6336
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 427s 854ms/step - accuracy: 0.6131 - loss: 0.6251 - val_accuracy: 0.6006 - val_loss: 0.6082
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 440s 851ms/step - accuracy: 0.6165 - loss: 0.5840 - val_accuracy: 0.5991 - val_loss: 0.6187
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 441s 849ms/step - accuracy: 0.6453 - loss: 0.5484 - val_accuracy: 0.7864 - val_loss: 0.5759
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 439s 844ms/step - accuracy: 0.8297 - loss: 0.4066 - val_accuracy: 0.8558 - val_loss: 0.3612


In [ ]:
loss, acc = model.evaluate(X_test_pad,y_test)
print("Test Accuracy:",acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 103ms/step - accuracy: 0.8619 - loss: 0.3499
Test Accuracy: 0.8618999719619751


In [1]:
def predict_sentiment(review):

  review = clean_text(review)

  seq = tokenizer.texts_to_sequences([review])

  padded = pad_sequences(seq,maxlen=max_len,padding='post')

  prediction = model.predict(padded)[0][0]

  print("\nReview:",review)
  print("Score:", prediction)

  if prediction >= 0.5:
    print("Sentiment: Positive")
  else:
    print("Sentiment: Negative")
